# Notebook 07 - LLM SFT Fine-Tuning (GPT-2)
# ----------------------------------------------------
 This notebook takes your processed forecasting dataset
 and turns it into an instruction-based dataset for
 training a small LLM (GPT-2) to explain sales trends,
 detect anomalies, and make marketing recommendations.
# ----------------------------------------------------


## 📌 1. Setup - Import Libraries

''''''http://localhost:8888/notebooks/notebooks/07_llm_pipeline_SFT_FineTuning_with_GPT2.ipynb#Import-Librariesimport datasets, transformers
print("datasets version:", datasets.__version__)
print("transformers version:", transformers.__version__)''''''

In [1]:
import os
from pathlib import Path
import json
import random
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset, Dataset
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

print("Notebook 07 Loaded")

/Users/imvenu/turing-ai-project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook 07 Loaded


## 📌 2. Paths

In [2]:
import pandas as pd
import numpy as np
import random
from pathlib import Path

# ======================================================
# SAVE LOCATION
# ======================================================
SFT_DATA_PATH = Path("../data/sft/sales_llm_instructions.csv")
SFT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

# ======================================================
# LOAD YOUR SALES DATA
# ======================================================
daily_path = Path("../data/processed/df_with_features.csv")
df_sales = pd.read_csv(daily_path, parse_dates=["datum"])

# Use only needed columns
category_cols = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
df_sales = df_sales[["datum"] + category_cols]

# ======================================================
# A. GENERAL INSIGHTS (behavior & clinic focused ONLY)
# ======================================================
general_instructions = [
    "Explain why weekends typically see higher pharmacy demand.",
    "Describe how staffing levels should change during peak days.",
    "Summarize factors that influence pharmacy sales patterns.",
    "Why do painkiller sales follow predictable weekly cycles?",
    "Explain how patient pickup behavior affects pharmacy demand.",
    "Describe how pharmacies should adjust inventory during peak periods.",
    "Summarize the main contributors to weekly demand variation.",
    "Explain when pharmacies should schedule additional replenishment.",
    "Give recommendations for aligning staffing with patient behavior.",
    "Explain how clinic appointment schedules influence pharmacy sales."
]

general_responses = [
    "Weekend demand is driven by patient pickup behavior following weekday clinic visits.",
    "Staffing should increase on peak days to manage higher prescription collection volumes.",
    "Sales patterns are shaped by refill cycles and clinic appointment schedules.",
    "Painkiller sales follow weekly routines linked to work schedules and clinic timings.",
    "Patient pickup habits strongly influence daily and weekly demand patterns.",
    "Inventory should be adjusted ahead of predictable pickup peaks to avoid stockouts.",
    "Weekly variation reflects patient routines, prescription renewals, and clinic flow.",
    "Replenishment should align with expected prescription collection peaks.",
    "Staffing should match predictable patient arrival patterns to maintain service quality.",
    "Clinic schedules directly impact when prescriptions are dispensed and collected."
]

general_records = []
for _ in range(40):
    general_records.append({
        "instruction": random.choice(general_instructions),
        "output": random.choice(general_responses)
    })

# ======================================================
# B. PATTERN-LEVEL INSIGHTS (NO seasonal language)
# ======================================================
pattern_templates = [
    "What weekly demand patterns exist for category {cat}?",
    "How does day-of-week influence sales for category {cat}?",
    "Explain demand behavior for category {cat} across the week.",
    "Why might {cat} peak after certain weekdays?",
    "Give staffing suggestions for handling {cat} demand patterns."
]

pattern_responses = [
    "This category follows weekly patterns driven by patient routines.",
    "Demand is typically higher after clinic-heavy weekdays.",
    "Sales reflect prescription refill cycles rather than seasonal effects.",
    "Peaks occur after workdays when patients collect prescriptions.",
    "Staffing should align with predictable pickup times for this category."
]

pattern_records = []
for _ in range(35):
    cat = random.choice(category_cols)
    pattern_records.append({
        "instruction": random.choice(pattern_templates).format(cat=cat),
        "output": random.choice(pattern_responses)
    })

# ======================================================
# C. DATA-ANCHORED ANALYTICAL SAMPLES (controlled output)
# ======================================================
data_records = []
for _, row in df_sales.head(200).iterrows():
    date = row["datum"].strftime("%Y-%m-%d")
    cat = random.choice(category_cols)
    value = float(row[cat])

    weekly_mean = df_sales[cat].mean()
    diff_pct = 0 if weekly_mean == 0 else (value - weekly_mean) / weekly_mean * 100

    inst = f"Analyze sales for {cat} on {date}."
    out = (
        f"On {date}, category {cat} recorded {value:.1f} units, "
        f"{abs(diff_pct):.1f}% {'above' if diff_pct > 0 else 'below'} typical weekly levels. "
        "Recommendation: adjust inventory based on refill patterns and align staffing "
        "with clinic-driven pickup schedules."
    )

    data_records.append({"instruction": inst, "output": out})

# ======================================================
# MERGE + SAVE
# ======================================================
full_dataset = pd.DataFrame(
    general_records + pattern_records + data_records
).sample(frac=1, random_state=42).reset_index(drop=True)

full_dataset.to_csv(SFT_DATA_PATH, index=False)

print("✔ Hybrid SFT dataset saved to:", SFT_DATA_PATH)
print("Total samples:", len(full_dataset))
full_dataset.head()

✔ Hybrid SFT dataset saved to: ../data/sft/sales_llm_instructions.csv
Total samples: 275


,instruction,output
0,Why do painkiller sales follow predictable wee...,Sales patterns are shaped by refill cycles and...
1,Analyze sales for R06 on 2014-04-06.,"On 2014-04-06, category R06 recorded 1.0 units..."
2,Analyze sales for M01AE on 2014-06-01.,"On 2014-06-01, category M01AE recorded 0.7 uni..."
3,Analyze sales for N02BE on 2014-03-24.,"On 2014-03-24, category N02BE recorded 23.6 un..."
4,Analyze sales for N05B on 2014-06-21.,"On 2014-06-21, category N05B recorded 12.0 uni..."


In [3]:
# =========================================
# 2. Paths (Corrected)
# =========================================

from pathlib import Path

BASE_DIR = Path("..")  # parent of notebooks/
DATA_DIR = BASE_DIR / "data"
SFT_DIR = DATA_DIR / "sft"           # <-- NEW: folder for SFT datasets
PROCESSED_DIR = DATA_DIR / "processed"
LLM_DIR = DATA_DIR / "llm"
MODEL_DIR = BASE_DIR / "models" / "sft_gpt2"

# Ensure folders exist
SFT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Path to SFT dataset
SFT_DATA_PATH = SFT_DIR / "sales_llm_instructions.csv"

print("Base directory        :", BASE_DIR.resolve())
print("Data directory        :", DATA_DIR.resolve())
print("SFT data directory    :", SFT_DIR.resolve())
print("Expected SFT CSV path :", SFT_DATA_PATH.resolve())
print("Model output dir      :", MODEL_DIR.resolve())

Base directory        : /Users/imvenu/turing-ai-project
Data directory        : /Users/imvenu/turing-ai-project/data
SFT data directory    : /Users/imvenu/turing-ai-project/data/sft
Expected SFT CSV path : /Users/imvenu/turing-ai-project/data/sft/sales_llm_instructions.csv
Model output dir      : /Users/imvenu/turing-ai-project/models/sft_gpt2


## 📌 3. Load Processed Feature Dataset

In [4]:
# =========================================
# 3. Load SFT Instruction Dataset
# =========================================

df = pd.read_csv(SFT_DATA_PATH)
df = df.sort_values("instruction")
print("Loaded dataset:", df.shape)
df.head()

Loaded dataset: (275, 2)


,instruction,output
189,Analyze sales for M01AB on 2014-02-07.,"On 2014-02-07, category M01AB recorded 2.7 uni..."
194,Analyze sales for M01AB on 2014-02-20.,"On 2014-02-20, category M01AB recorded 7.0 uni..."
264,Analyze sales for M01AB on 2014-02-25.,"On 2014-02-25, category M01AB recorded 5.0 uni..."
240,Analyze sales for M01AB on 2014-03-05.,"On 2014-03-05, category M01AB recorded 3.3 uni..."
105,Analyze sales for M01AB on 2014-03-07.,"On 2014-03-07, category M01AB recorded 4.0 uni..."


## 📌 4. Build Instruction Dataset Automatically

We’re building several types of instructions:

🔹 Trend explanation

🔹 Anomaly detection

🔹 Weekday/weekend sales insights

🔹 Rolling-window behavior

🔹 Inventory recommendations

In [5]:
def build_instruction(row):
    date = row["datum"].strftime("%Y-%m-%d")
    sales = row["N02BE"]  # your main sales category

    instructions = [
        f"Explain the sales trend on {date}.",
        f"Are the sales on {date} unusually high or low?",
        f"What marketing action should be taken based on sales of {sales} units on {date}?",
        f"Describe the weekly pattern around {date}.",
        f"Give a short inventory recommendation for the day {date}."
    ]
    return random.choice(instructions)

## 📌 5. Build Output (Response) Automatically

Using your engineered features.

''''''def build_response(row):
    weekday = row["Weekday_Name"]
    sales = row["N02BE"]
    roll7 = row["N02BE_roll7"]
    roll30 = row["N02BE_roll30"]

    trend = "increasing" if roll7 > roll30 else "decreasing"

    resp = (
        f"On {row['datum'].strftime('%Y-%m-%d')}, sales were {sales} units. "
        f"This day is a {weekday}, which typically has an average of {row['Weekday_Avg']:.1f} units. "
        f"The 7-day trend shows {trend} demand. "
        f"Based on this, inventory should be adjusted to maintain optimal stock."
    )
    return resp'''''''

In [6]:
def build_instruction(row):
    """
    Your dataset already contains the full instruction string.
    Just return it directly.
    """
    return row["instruction"]

def build_response(row):
    """
    Your dataset already contains the output text.
    """
    return row["output"]

## 📌 6. Create SFT JSONL File

In [7]:
import json
from tqdm import tqdm

sft_records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    record = {
        "instruction": row["instruction"],
        "input": "",
        "output": row["output"]
    }
    sft_records.append(record)

output_path = LLM_DIR / "sft_dataset.jsonl"

with open(output_path, "w") as f:
    for r in sft_records:
        f.write(json.dumps(r) + "\n")

print("SFT dataset saved to:", output_path)
print("Total records:", len(sft_records))

100%|██████████████████████████████████████| 275/275 [00:00<00:00, 75694.55it/s]

SFT dataset saved to: ../data/llm/sft_dataset.jsonl
Total records: 275


## 📌 7. Load Dataset into HuggingFace Format

In [8]:
ds = load_dataset("json", data_files=str(output_path), split="train")
ds

Generating train split: 275 examples [00:00, 62956.91 examples/s]


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 275
})

## 📌 8. Load GPT-2 Tokenizer + Model

In [9]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # important for batching

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

Embedding(50257, 768)

## 📌 9. Tokenize

In [10]:
def tokenize(batch):
    # Create combined instruction-output strings
    texts = [
        instr + tokenizer.eos_token + out + tokenizer.eos_token
        for instr, out in zip(batch["instruction"], batch["output"])
    ]
    
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [11]:
tokenized_ds = ds.map(tokenize, batched=True)

Map: 100%|██████████████████████████| 275/275 [00:00<00:00, 12538.68 examples/s]


## 📌 10. Training Arguments

In [12]:
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    warmup_steps=50,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

## 📌 11. Train GPT-2 (SFT)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator
)

trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("Model saved at:", MODEL_DIR)

/Users/imvenu/turing-ai-project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,4.616400
40,2.039200
60,0.850000
80,0.840900
100,0.570200
120,0.501900


Model saved at: ../models/sft_gpt2


## 📌 12. Quick Evaluation (Chat-style)

In [14]:
import torch
import re

def ask(prompt, model, tokenizer, device="mps"):
    """
    STRICT inference mode for healthcare operations text.
    Designed for GPT-2 limitations.
    """
    model.to(device)
    model.eval()

    full_prompt = (
        "You are a healthcare operations analyst.\n"
        "Write a concise operational explanation.\n\n"
        "Rules:\n"
        "- Use plain text only\n"
        "- No symbols or special characters\n"
        "- No references to systems, tools, software, or policies\n"
        "- No examples, names, or dates\n"
        "- Neutral, analytical tone\n"
        "- Maximum two sentences\n\n"
        "Question:\n"
        f"{prompt}\n\n"
        "Answer:\n"
    )

    inputs = tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            min_new_tokens=20,
            max_new_tokens=150,      # HARD cap
            do_sample=True,
            temperature=0.25,       # low creativity
            top_p=0.6,
            repetition_penalty=1.7,
            no_repeat_ngram_size=7,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Strip prompt echoes
    if "Answer:" in text:
        text = text.split("Answer:", 1)[1].strip()

    # Hard sanitation (GPT-2 safety net)
    text = re.sub(r"[^a-zA-Z0-9.,\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    print(text)
    return text

In [23]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from pathlib import Path

# Device
device = "mps" if torch.backends.mps.is_available() else "cpu"

# Load model path
MODEL_DIR = Path("../outputs/llm/gpt2-sales-sft")  # or RLHF if available

# Tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Model
model = GPT2LMHeadModel.from_pretrained(str(MODEL_DIR))
model.to(device)
model.eval()

print("✅ Model ready on", device)

✅ Model ready on mps


In [24]:
print(type(model))
print(model.device)
print(tokenizer is not None)

<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
mps:0
True


In [25]:
import torch
import re

def ask(prompt, model, tokenizer, device="mps"):
    model.to(device)
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.5,
            top_p=0.8,
            repetition_penalty=1.4,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    text = text.strip()

    print(text)
    return text

## Save SFT Model

In [26]:
# ==========================
# FORCE SAVE SFT MODEL (robust)
# ==========================

from pathlib import Path

SFT_MODEL_DIR = Path("../outputs/llm/gpt2-sales-sft")
SFT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save model + tokenizer explicitly
trainer.save_model(str(SFT_MODEL_DIR))
tokenizer.save_pretrained(str(SFT_MODEL_DIR))

# Safety check
files = [f.name for f in SFT_MODEL_DIR.iterdir()]
print("Saved files:", files)

assert any(
    f in files for f in ["pytorch_model.bin", "model.safetensors"]
), "❌ Model weights not saved!"
print("✅ SFT model successfully saved")

Saved files: ['model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'config.json', 'tokenizer.json', 'generation_config.json', 'merges.txt', 'training_args.bin', 'vocab.json']
✅ SFT model successfully saved


## Pre-Flight Check Cell (Run before Notebook 08)

In [27]:
from pathlib import Path

def check_model_dir(path: Path):
    if not path.exists():
        return False, ["directory missing"]

    files = {f.name for f in path.iterdir()}
    required = {"config.json"}
    weight_files = {"pytorch_model.bin", "model.safetensors"}

    missing = []
    if not required.issubset(files):
        missing.append("config.json")
    if not files.intersection(weight_files):
        missing.append("model weights (.bin or .safetensors)")

    return len(missing) == 0, missing


ok, missing = check_model_dir(Path("../outputs/llm/gpt2-sales-sft"))
print("SFT model OK:", ok)
print("Missing:", missing)

SFT model OK: True
Missing: []


In [28]:
from pathlib import Path

print("🔍 PRE-FLIGHT CHECK — LLM PIPELINE\n")

# --------------------------------------------------
# Define expected paths
# --------------------------------------------------
BASE_DIR = Path("..")

DATA_SFT_DIR = BASE_DIR / "data" / "sft"
SFT_DATA_FILE = DATA_SFT_DIR / "sales_llm_instructions.csv"

SFT_MODEL_DIR = BASE_DIR / "outputs" / "llm" / "gpt2-sales-sft"
RLHF_MODEL_DIR = BASE_DIR / "outputs" / "llm" / "gpt2-sales-rlhf"

EXPECTED_MODEL_FILES = [
    "config.json",
    "tokenizer.json"
]

WEIGHT_FILES = [
    "pytorch_model.bin",
    "model.safetensors"
]

# --------------------------------------------------
# Helper function
# --------------------------------------------------
def check_path(path, is_dir=False):
    if path.exists():
        print(f"✅ EXISTS: {path}")
        if is_dir:
            return True
        return True
    else:
        print(f"❌ MISSING: {path}")
        return False

# --------------------------------------------------
# 1. Check SFT dataset
# --------------------------------------------------
print("\n📁 Checking SFT dataset")
sft_data_ok = check_path(SFT_DATA_FILE)

# --------------------------------------------------
# 2. Check SFT model directory and files
# --------------------------------------------------
print("\n🧠 Checking SFT model directory")
sft_model_ok = check_path(SFT_MODEL_DIR, is_dir=True)

missing_sft_files = []

if sft_model_ok:

    # Check basic required files
    for f in ["config.json", "tokenizer.json"]:
        if not (SFT_MODEL_DIR / f).exists():
            missing_sft_files.append(f)

    # Check weight file (either .bin OR .safetensors is valid)
    if not any((SFT_MODEL_DIR / wf).exists() for wf in ["pytorch_model.bin", "model.safetensors"]):
        missing_sft_files.append("model weights (.bin or .safetensors)")

    if missing_sft_files:
        print("❌ Missing SFT model files:")
        for f in missing_sft_files:
            print("   -", f)
        sft_model_ok = False
    else:
        print("✅ All required SFT model files found")

# --------------------------------------------------
# 3. Check RLHF model directory (optional but expected)
# --------------------------------------------------
print("\n🧠 Checking RLHF model directory")
rlhf_model_ok = check_path(RLHF_MODEL_DIR, is_dir=True)

if rlhf_model_ok:
    missing_rlhf_files = []
    for f in EXPECTED_MODEL_FILES:
        if not (RLHF_MODEL_DIR / f).exists():
            missing_rlhf_files.append(f)

    if missing_rlhf_files:
        print("⚠️ RLHF directory exists but is incomplete:")
        for f in missing_rlhf_files:
            print("   -", f)
        print("   (This is OK if RLHF training has not run yet)")
    else:
        print("✅ All required RLHF model files found")

# --------------------------------------------------
# Final verdict
# --------------------------------------------------
print("\n📊 FINAL STATUS")
if sft_data_ok and sft_model_ok:
    print("✅ SAFE TO PROCEED to Notebook 08 (RLHF / evaluation)")
else:
    print("❌ DO NOT PROCEED — fix the missing components above")

🔍 PRE-FLIGHT CHECK — LLM PIPELINE


📁 Checking SFT dataset
✅ EXISTS: ../data/sft/sales_llm_instructions.csv

🧠 Checking SFT model directory
✅ EXISTS: ../outputs/llm/gpt2-sales-sft
✅ All required SFT model files found

🧠 Checking RLHF model directory
✅ EXISTS: ../outputs/llm/gpt2-sales-rlhf
⚠️ RLHF directory exists but is incomplete:
   - config.json
   - tokenizer.json
   (This is OK if RLHF training has not run yet)

📊 FINAL STATUS
✅ SAFE TO PROCEED to Notebook 08 (RLHF / evaluation)
